In [3]:
import chromadb
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
import os

In [4]:
# --- CONFIGURATION ---
DATA_FOLDER = "data"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100
DB_NAME = "my_documents"

# 1. Setup Vector DB Client
print("🔌 Connecting to Vector Database...")
client = chromadb.PersistentClient(path="my_vector_db")
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# --- CLEAN DATABASE COMMAND ---
# This fixes the "Old Data" bugs by letting you wipe the slate clean.
print("\n" + "="*40)
print("   🗑️  DATABASE MANAGEMENT")
print("="*40)
reset_choice = input("Do you want to WIPE the database and start fresh? (y/n): ")

if reset_choice.lower() == 'y':
    try:
        client.delete_collection(name=DB_NAME)
        print("🧹 Database wiped clean! All old memories deleted.")
    except Exception as e:
        print(f"⚠️  Could not delete collection (maybe it didn't exist yet). Continuing...")

# Create (or recreate) the collection
collection = client.get_or_create_collection(name=DB_NAME, embedding_function=embedding_func)
print(f"✅ Active Collection: '{DB_NAME}'\n")

# 2. Check Data Folder
if not os.path.exists(DATA_FOLDER):
    os.makedirs(DATA_FOLDER)
    print(f"❌ Created folder '{DATA_FOLDER}'. Please put your PDFs inside it and run this again.")
    exit()

pdf_files = [f for f in os.listdir(DATA_FOLDER) if f.endswith('.pdf')]
if not pdf_files:
    print(f"❌ No PDF files found in '{DATA_FOLDER}'.")
    exit()

print(f"📚 Found {len(pdf_files)} PDFs. Starting ingestion...")

# 3. Setup Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", " ", ""]
)

total_chunks = 0

# 4. Loop & Ingest
for filename in pdf_files:
    file_path = os.path.join(DATA_FOLDER, filename)
    print(f"\n📄 Processing: {filename}...")
    
    try:
        # Extract Text
        reader = PdfReader(file_path)
        full_text = ""
        for page in reader.pages:
            text = page.extract_text()
            if text:
                full_text += text + "\n"
        
        # Chunk Text
        chunks = text_splitter.split_text(full_text)
        
        if not chunks:
            print("   ⚠️  Warning: No text extracted. Is this an image-only PDF?")
            continue

        # Add Metadata (Crucial for the "Source" feature)
        metadatas = [{"source": filename} for _ in chunks]
        ids = [f"{filename}_chunk_{i}" for i in range(len(chunks))]

        # Save to DB
        collection.add(
            documents=chunks,
            metadatas=metadatas,
            ids=ids
        )
        total_chunks += len(chunks)
        print(f"   -> Added {len(chunks)} chunks.")
            
    except Exception as e:
        print(f"   ⚠️ Error processing {filename}: {e}")

print(f"\n✅ MISSION COMPLETE. Stored a total of {total_chunks} chunks from {len(pdf_files)} books.")

🔌 Connecting to Vector Database...

   🗑️  DATABASE MANAGEMENT
⚠️  Could not delete collection (maybe it didn't exist yet). Continuing...
✅ Active Collection: 'my_documents'

📚 Found 7 PDFs. Starting ingestion...

📄 Processing: 4107d441-0646-469d-832f-080ac76a2bd7.pdf.pdf...
   -> Added 377 chunks.

📄 Processing: A-Self-Care-Guide-Care-2-Caregivers.pdf...
   -> Added 17 chunks.

📄 Processing: AtomicHabit.pdf...
   -> Added 517 chunks.

📄 Processing: data.pdf...
   -> Added 99 chunks.

📄 Processing: Deep Work PDF.pdf...
   -> Added 74 chunks.

📄 Processing: The-Happiness-Trap-Harris-R1.pdf...
   -> Added 548 chunks.

📄 Processing: Your-Toolkit-for-Self-Acceptance-and-Positive-Change-in-2024-Newport-Healthcare.pdf...
   -> Added 11 chunks.

✅ MISSION COMPLETE. Stored a total of 1643 chunks from 7 books.
